# 16. Heuristiken: Lösungen schrittweise verbessern

Bei manchen Aufgaben gibt es so viele Möglichkeiten, dass man nicht alle durchprobieren kann ([Notebook 1](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf1/01_durchprobieren.ipynb)),
und ein gieriges Verfahren ([Notebook 12](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf1/12_greedy.ipynb)) liefert nur eine **brauchbare**, aber nicht die beste Lösung. Dann hilft oft eine **Heuristik**:
ein Verfahren, das **ohne Garantie** eine **gute** Lösung sucht. Die Grundidee ist einfach: Man startet mit **irgendeiner** Lösung, **bewertet** sie mit einer Zahl und
**ändert sie ein wenig**. Ist das Ergebnis besser, behält man es. Das wiederholt man, bis nichts mehr besser wird oder die Zeit abgelaufen ist.

In diesem Notebook lernst du:

- eine Lösung durch eine **Bewertungsfunktion** (ein Zahlenwert, den man klein oder groß haben will) zu beurteilen und ihre **Nachbarn** (kleine Änderungen) zu erzeugen,
- das **Hill-Climbing** (nur Verbesserungen annehmen) und warum es in einem **lokalen Optimum** hängen bleibt,
- **Zufallsneustarts** mit festem Startwert (wie in [Notebook 15](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf1/15_ereignisse_zufall.ipynb)) und **Simulated Annealing**, das Verschlechterungen mit
  sinkender Wahrscheinlichkeit zulässt,
- ehrlich mit dem **echten Optimum** zu vergleichen, wo man es durch Durchprobieren kennt, und in der Dokumentation zu sagen, **wie gut** das Ergebnis ist (und dass es **keine Garantie** gibt).

In den Wettbewerbs-Ecken verteilst du Lochmuster auf Schlüsselkarten (nach Aufgabe „Schlüssellöcher“) und setzt ein Buchstabenrätsel zusammen (nach Aufgabe „Wortsuche“).

## 1. Bewertung und Nachbarn

Unser Beispiel: 14 Gepäckstücke sollen auf **zwei Wagen** verteilt werden, und zwar so, dass die beiden Wagen **möglichst gleich schwer** beladen sind. Die Gewichte (in kg):

```python
gewichte = [25, 47, 44, 18, 33, 48, 40, 50, 47, 14, 48, 10, 40, 26]
```

Eine **Lösung** ist eine Liste `gruppe` mit einer 0 oder 1 für jedes Gepäckstück: `gruppe[i] = 0` heißt „Gepäckstück `i` liegt auf Wagen 0“, bei 1 liegt es auf Wagen 1.
Die **Bewertung** ist der **Unterschied** der beiden Wagengewichte (Betrag der Differenz). Je kleiner, desto besser, 0 wäre perfekt. Die Gesamtmasse ist 490 kg.
Eine **kleine Änderung** ist: **ein** Gepäckstück auf den anderen Wagen legen. Die Lösungen, die man so aus `gruppe` erhält, heißen **Nachbarn** von `gruppe`. Es gibt bei 14 Stücken genau 14 Nachbarn.

#### Aufgabe 1.1

Schreibe `unterschied(gewichte, gruppe)`: Sie gibt den Betrag der Differenz der beiden Wagengewichte zurück (`abs(...)` liefert den Betrag).

**Erwartete Ausgabe:**
```
490
64
20
```

In [ ]:
gewichte = [25, 47, 44, 18, 33, 48, 40, 50, 47, 14, 48, 10, 40, 26]

def unterschied(gewichte, gruppe):
    # dein Code
    return 0

print(unterschied(gewichte, [0] * 14))
print(unterschied(gewichte, [0, 1] * 7))
print(unterschied(gewichte, [0] * 7 + [1] * 7))

<details>
<summary>Tipp anzeigen</summary>

Zwei Summen `wagen0` und `wagen1`, eine Schleife über alle Stellen `i`: Ist `gruppe[i] == 0`, kommt `gewichte[i]` zu `wagen0`, sonst zu `wagen1`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
gewichte = [25, 47, 44, 18, 33, 48, 40, 50, 47, 14, 48, 10, 40, 26]

def unterschied(gewichte, gruppe):
    wagen0 = 0
    wagen1 = 0
    for i in range(len(gewichte)):
        if gruppe[i] == 0:
            wagen0 = wagen0 + gewichte[i]
        else:
            wagen1 = wagen1 + gewichte[i]
    return abs(wagen0 - wagen1)

print(unterschied(gewichte, [0] * 14))
print(unterschied(gewichte, [0, 1] * 7))
print(unterschied(gewichte, [0] * 7 + [1] * 7))
```

Liegt alles auf Wagen 0, ist der Unterschied die ganze Masse (490). Bei abwechselnder Verteilung (`[0, 1] * 7`) und bei „erste sieben auf Wagen 0, letzte sieben auf Wagen 1“ sind es 64 und 20.

</details>

#### Frage 1.2

1. Wie viele verschiedene Verteilungen der 14 Stücke gibt es?
2. Wie viele wären es bei 40 Stücken? Wie lange würde das Durchprobieren dauern, wenn dein Programm eine Million Verteilungen pro Sekunde bewertet?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Jedes Stück hat zwei Möglichkeiten: `2^14 =` **16 384** Verteilungen. Das ist zum Durchprobieren noch gut zu schaffen.
2. `2^40 =` **1 099 511 627 776**, rund 1,1 Billionen. Bei einer Million pro Sekunde wären das etwa 1 100 000 Sekunden, also rund **12,7 Tage**. Hier braucht man ein Verfahren, das **nicht alles** probiert.
   (Bei einem Rucksackproblem mit ganzen Zahlen gäbe es auch die Tabellen aus [Notebook 11](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf1/11_tabellen_dp.ipynb), solange die Gewichte klein sind. Hier üben wir das **allgemeine** Vorgehen.)

</details>

### Hill-Climbing: immer zum besten Nachbarn

Das einfachste Verfahren heißt **Hill-Climbing** (Bergsteigen, hier bergab zum kleinsten Unterschied): Man bewertet alle Nachbarn und geht zum **besten**, **wenn er besser ist** als die aktuelle Lösung. Gibt es keinen besseren
Nachbarn, hört man auf.

#### Aufgabe 1.3

`unterschied` ist fertig. Schreibe `bergauf(gewichte, gruppe)`: Sie gibt `(gruppe, schritte)` zurück, die verbesserte Verteilung und die Zahl der Schritte. In jedem Schritt probierst du **alle** 14 Nachbarn:
Stück `i` umlegen (`gruppe[i] = 1 - gruppe[i]`), bewerten und **wieder zurücklegen**. Merke dir den kleinsten Wert und das zugehörige Stück. Nur wenn er **kleiner** ist als der aktuelle Unterschied, legst du dieses Stück
wirklich um und machst weiter. Sonst ist Schluss. Die Ausgabe startet mit allen Stücken auf Wagen 0.

**Erwartete Ausgabe:**
```
Start: 490
Ende:  10 nach 5 Schritten
```

In [ ]:
gewichte = [25, 47, 44, 18, 33, 48, 40, 50, 47, 14, 48, 10, 40, 26]

def unterschied(gewichte, gruppe):
    wagen0 = 0
    wagen1 = 0
    for i in range(len(gewichte)):
        if gruppe[i] == 0:
            wagen0 = wagen0 + gewichte[i]
        else:
            wagen1 = wagen1 + gewichte[i]
    return abs(wagen0 - wagen1)

def bergauf(gewichte, gruppe):
    gruppe = gruppe[:]                                  # Kopie, die Liste des Aufrufers bleibt unverändert
    schritte = 0
    while True:
        bester = unterschied(gewichte, gruppe)
        bester_teil = -1                                # -1: noch kein besserer Nachbar gefunden
        # dein Code: alle Nachbarn probieren, besten Wert und sein Stück merken
        if bester_teil == -1:                           # kein Nachbar ist besser: fertig
            break
        gruppe[bester_teil] = 1 - gruppe[bester_teil]   # zum besten Nachbarn gehen
        schritte = schritte + 1
    return gruppe, schritte

start = [0] * len(gewichte)
ende, schritte = bergauf(gewichte, start)
print('Start:', unterschied(gewichte, start))
print('Ende: ', unterschied(gewichte, ende), 'nach', schritte, 'Schritten')

<details>
<summary>Tipp anzeigen</summary>

`for i in range(len(gewichte)):` Stück `i` umlegen, `wert = unterschied(gewichte, gruppe)`, wieder zurücklegen. Ist `wert < bester`, dann `bester = wert` und `bester_teil = i`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
gewichte = [25, 47, 44, 18, 33, 48, 40, 50, 47, 14, 48, 10, 40, 26]

def unterschied(gewichte, gruppe):
    wagen0 = 0
    wagen1 = 0
    for i in range(len(gewichte)):
        if gruppe[i] == 0:
            wagen0 = wagen0 + gewichte[i]
        else:
            wagen1 = wagen1 + gewichte[i]
    return abs(wagen0 - wagen1)

def bergauf(gewichte, gruppe):
    gruppe = gruppe[:]                                  # Kopie, die Liste des Aufrufers bleibt unverändert
    schritte = 0
    while True:
        bester = unterschied(gewichte, gruppe)
        bester_teil = -1                                # -1: noch kein besserer Nachbar gefunden
        for i in range(len(gewichte)):
            gruppe[i] = 1 - gruppe[i]                   # Nachbar herstellen: Stück i umlegen
            wert = unterschied(gewichte, gruppe)
            gruppe[i] = 1 - gruppe[i]                   # und gleich wieder zurücklegen
            if wert < bester:
                bester = wert
                bester_teil = i
        if bester_teil == -1:                           # kein Nachbar ist besser: fertig
            break
        gruppe[bester_teil] = 1 - gruppe[bester_teil]   # zum besten Nachbarn gehen
        schritte = schritte + 1
    return gruppe, schritte

start = [0] * len(gewichte)
ende, schritte = bergauf(gewichte, start)
print('Start:', unterschied(gewichte, start))
print('Ende: ', unterschied(gewichte, ende), 'nach', schritte, 'Schritten')
```

Das Verfahren startet bei Unterschied 490 und braucht nur **5 Schritte** (jeweils mit 14 Bewertungen) bis zu einem Unterschied von **10**. Mit rund 90 Bewertungen
statt 16 384 ist das sehr schnell. Aber ist 10 gut? Das prüfen wir gleich.

</details>

In [ ]:
gewichte = [25, 47, 44, 18, 33, 48, 40, 50, 47, 14, 48, 10, 40, 26]

def unterschied(gewichte, gruppe):
    wagen0 = 0
    wagen1 = 0
    for i in range(len(gewichte)):
        if gruppe[i] == 0:
            wagen0 = wagen0 + gewichte[i]
        else:
            wagen1 = wagen1 + gewichte[i]
    return abs(wagen0 - wagen1)

import itertools

def optimum(gewichte):
    """Kleinster möglicher Unterschied, alle 2^n Verteilungen durchprobiert (nur für kleines n)."""
    bester = sum(gewichte)
    for gruppe in itertools.product([0, 1], repeat=len(gewichte)):
        wert = unterschied(gewichte, gruppe)
        if wert < bester:
            bester = wert
    return bester

print(optimum(gewichte))

Die Zelle probiert wie in [Notebook 1](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf1/01_durchprobieren.ipynb) alle 16 384 Verteilungen durch (`itertools.product([0, 1], repeat=14)`). Das Ergebnis: Es gibt eine Verteilung mit
Unterschied **0**. Unser Hill-Climbing hat bei **10** aufgehört. Es steckt in einem **lokalen Optimum**.

#### Frage 1.4

Das Hill-Climbing hört bei Unterschied 10 auf, obwohl es Lösungen mit Unterschied 0 gibt.

1. Warum hört es auf? Was gilt für alle 14 Nachbarn der Endlösung?
2. Was heißt **lokales Optimum**, was **globales Optimum**?
3. Wie könnte man die Verteilung mit Unterschied 0 trotzdem erreichen, ohne alles durchzuprobieren? Sammle zwei Ideen.

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Kein Nachbar ist **besser** als die Endlösung (jeder der 14 Nachbarn hat Unterschied 10 oder mehr). Das Verfahren geht nur zu besseren Nachbarn, also bleibt es stehen.
2. Ein **lokales Optimum** ist eine Lösung, die besser ist als **alle ihre Nachbarn**. Ein **globales Optimum** ist eine beste Lösung überhaupt. Jedes globale Optimum ist auch ein lokales, aber nicht umgekehrt.
3. (a) **Anders starten**: Mit einer anderen Anfangsverteilung (zufällig!) landet man vielleicht in einem anderen, besseren lokalen Optimum. Das ist der **Neustart** (Abschnitt 2).
   (b) **Auch Verschlechterungen erlauben**: Wer manchmal einen schlechteren Nachbarn annimmt, kann aus einem lokalen Optimum herauslaufen. Das ist **Simulated Annealing** (Abschnitt 3).
   (Eine dritte Idee: größere Änderungen als Nachbarn zulassen, zum Beispiel **zwei** Stücke tauschen.)

</details>

## 2. Zufall mit Neustart

**Erste Idee: eine gute Startlösung.** In [Notebook 12](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf1/12_greedy.ipynb) hast du gierige Verfahren kennengelernt. Hier wäre das: Die Stücke der Größe nach sortieren (das schwerste zuerst) und jedes auf den
**leichteren** Wagen legen. Danach versuchen wir, diese Startlösung mit Hill-Climbing zu verbessern.

#### Aufgabe 2.1

Die gierige Funktion `gierig` und `bergauf` sind fertig. Setze den Aufruf so zusammen, dass `bergauf` mit der gierigen Startlösung läuft. Lies in der Lösung nach, wie der Code aussieht,
und beantworte dann die Frage unter der Ausgabe.

**Erwartete Ausgabe:**
```
gierig: 2
danach: 2 nach 0 Schritten
```

In [ ]:
gewichte = [25, 47, 44, 18, 33, 48, 40, 50, 47, 14, 48, 10, 40, 26]

def unterschied(gewichte, gruppe):
    wagen0 = 0
    wagen1 = 0
    for i in range(len(gewichte)):
        if gruppe[i] == 0:
            wagen0 = wagen0 + gewichte[i]
        else:
            wagen1 = wagen1 + gewichte[i]
    return abs(wagen0 - wagen1)

def bergauf(gewichte, gruppe):
    gruppe = gruppe[:]                                  # Kopie, die Liste des Aufrufers bleibt unverändert
    schritte = 0
    while True:
        bester = unterschied(gewichte, gruppe)
        bester_teil = -1                                # -1: noch kein besserer Nachbar gefunden
        for i in range(len(gewichte)):
            gruppe[i] = 1 - gruppe[i]                   # Nachbar herstellen: Stück i umlegen
            wert = unterschied(gewichte, gruppe)
            gruppe[i] = 1 - gruppe[i]                   # und gleich wieder zurücklegen
            if wert < bester:
                bester = wert
                bester_teil = i
        if bester_teil == -1:                           # kein Nachbar ist besser: fertig
            break
        gruppe[bester_teil] = 1 - gruppe[bester_teil]   # zum besten Nachbarn gehen
        schritte = schritte + 1
    return gruppe, schritte

def gierig(gewichte):
    gruppe = [0] * len(gewichte)
    wagen0 = 0
    wagen1 = 0
    for i in sorted(range(len(gewichte)), key=lambda i: -gewichte[i]):     # die schwersten zuerst
        if wagen0 <= wagen1:
            gruppe[i] = 0
            wagen0 = wagen0 + gewichte[i]
        else:
            gruppe[i] = 1
            wagen1 = wagen1 + gewichte[i]
    return gruppe

start = gierig(gewichte)
print('gierig:', unterschied(gewichte, start))
# dein Code: bergauf mit start aufrufen und das Ergebnis ausgeben

<details>
<summary>Tipp anzeigen</summary>

`ende, schritte = bergauf(gewichte, start)`, dann `unterschied(gewichte, ende)` ausgeben.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
gewichte = [25, 47, 44, 18, 33, 48, 40, 50, 47, 14, 48, 10, 40, 26]

def unterschied(gewichte, gruppe):
    wagen0 = 0
    wagen1 = 0
    for i in range(len(gewichte)):
        if gruppe[i] == 0:
            wagen0 = wagen0 + gewichte[i]
        else:
            wagen1 = wagen1 + gewichte[i]
    return abs(wagen0 - wagen1)

def bergauf(gewichte, gruppe):
    gruppe = gruppe[:]                                  # Kopie, die Liste des Aufrufers bleibt unverändert
    schritte = 0
    while True:
        bester = unterschied(gewichte, gruppe)
        bester_teil = -1                                # -1: noch kein besserer Nachbar gefunden
        for i in range(len(gewichte)):
            gruppe[i] = 1 - gruppe[i]                   # Nachbar herstellen: Stück i umlegen
            wert = unterschied(gewichte, gruppe)
            gruppe[i] = 1 - gruppe[i]                   # und gleich wieder zurücklegen
            if wert < bester:
                bester = wert
                bester_teil = i
        if bester_teil == -1:                           # kein Nachbar ist besser: fertig
            break
        gruppe[bester_teil] = 1 - gruppe[bester_teil]   # zum besten Nachbarn gehen
        schritte = schritte + 1
    return gruppe, schritte

def gierig(gewichte):
    gruppe = [0] * len(gewichte)
    wagen0 = 0
    wagen1 = 0
    for i in sorted(range(len(gewichte)), key=lambda i: -gewichte[i]):     # die schwersten zuerst
        if wagen0 <= wagen1:
            gruppe[i] = 0
            wagen0 = wagen0 + gewichte[i]
        else:
            gruppe[i] = 1
            wagen1 = wagen1 + gewichte[i]
    return gruppe

start = gierig(gewichte)
print('gierig:', unterschied(gewichte, start))
ende, schritte = bergauf(gewichte, start)
print('danach:', unterschied(gewichte, ende), 'nach', schritte, 'Schritten')
```

Die gierige Startlösung hat schon nur den Unterschied **2**, und das Hill-Climbing findet **keinen besseren** Nachbarn (0 Schritte). Die Lösung ist ein lokales Optimum, das globale Optimum ist **0**.
Auch eine gute Startlösung schützt also nicht davor, in einem lokalen Optimum zu landen.

</details>

**Zweite Idee: viele zufällige Starts.** Man startet das Hill-Climbing **mehrfach** mit jeweils einer **zufälligen** Startverteilung und merkt sich das **beste** Ergebnis. Das ist ein **Zufallsneustart**.
Wie in [Notebook 15](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf1/15_ereignisse_zufall.ipynb) nimmst du `random.Random(startwert)` und setzt den Startwert **einmal** am Anfang: Dann sind die Ergebnisse bei jedem Lauf dieselben, und du kannst sie in
der Dokumentation nennen. Eine zufällige Startverteilung ist `[zufall.randint(0, 1) for teil in gewichte]`.

#### Aufgabe 2.2

Schreibe `neustarts(gewichte, versuche, startwert)`: Sie erzeugt **einmal** `zufall = random.Random(startwert)` und macht dann `versuche`-mal: zufällige Startverteilung, `bergauf`, Unterschied berechnen.
Zurück gibt sie den **kleinsten** Unterschied aller Versuche. Die Ausgabe zeigt für 1, 3, 10 und 30 Neustarts die Ergebnisse bei den Startwerten 0 bis 7.

**Erwartete Ausgabe:**
```
1 Neustarts: [4, 2, 2, 10, 2, 2, 14, 6]
3 Neustarts: [2, 0, 2, 2, 0, 2, 2, 4]
10 Neustarts: [0, 0, 2, 2, 0, 2, 2, 0]
30 Neustarts: [0, 0, 0, 0, 0, 0, 0, 0]
```

In [ ]:
gewichte = [25, 47, 44, 18, 33, 48, 40, 50, 47, 14, 48, 10, 40, 26]

def unterschied(gewichte, gruppe):
    wagen0 = 0
    wagen1 = 0
    for i in range(len(gewichte)):
        if gruppe[i] == 0:
            wagen0 = wagen0 + gewichte[i]
        else:
            wagen1 = wagen1 + gewichte[i]
    return abs(wagen0 - wagen1)

def bergauf(gewichte, gruppe):
    gruppe = gruppe[:]                                  # Kopie, die Liste des Aufrufers bleibt unverändert
    schritte = 0
    while True:
        bester = unterschied(gewichte, gruppe)
        bester_teil = -1                                # -1: noch kein besserer Nachbar gefunden
        for i in range(len(gewichte)):
            gruppe[i] = 1 - gruppe[i]                   # Nachbar herstellen: Stück i umlegen
            wert = unterschied(gewichte, gruppe)
            gruppe[i] = 1 - gruppe[i]                   # und gleich wieder zurücklegen
            if wert < bester:
                bester = wert
                bester_teil = i
        if bester_teil == -1:                           # kein Nachbar ist besser: fertig
            break
        gruppe[bester_teil] = 1 - gruppe[bester_teil]   # zum besten Nachbarn gehen
        schritte = schritte + 1
    return gruppe, schritte

import random

def neustarts(gewichte, versuche, startwert):
    zufall = random.Random(startwert)                   # EINMAL, vor allen Versuchen
    bester = sum(gewichte)
    # dein Code
    return bester

for versuche in [1, 3, 10, 30]:
    werte = [neustarts(gewichte, versuche, startwert) for startwert in range(8)]
    print(versuche, 'Neustarts:', werte)

<details>
<summary>Tipp anzeigen</summary>

Eine Schleife `for versuch in range(versuche):`. Darin `start = [zufall.randint(0, 1) for teil in gewichte]`, `ende, schritte = bergauf(gewichte, start)`, dann den Unterschied mit `bester` vergleichen.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
gewichte = [25, 47, 44, 18, 33, 48, 40, 50, 47, 14, 48, 10, 40, 26]

def unterschied(gewichte, gruppe):
    wagen0 = 0
    wagen1 = 0
    for i in range(len(gewichte)):
        if gruppe[i] == 0:
            wagen0 = wagen0 + gewichte[i]
        else:
            wagen1 = wagen1 + gewichte[i]
    return abs(wagen0 - wagen1)

def bergauf(gewichte, gruppe):
    gruppe = gruppe[:]                                  # Kopie, die Liste des Aufrufers bleibt unverändert
    schritte = 0
    while True:
        bester = unterschied(gewichte, gruppe)
        bester_teil = -1                                # -1: noch kein besserer Nachbar gefunden
        for i in range(len(gewichte)):
            gruppe[i] = 1 - gruppe[i]                   # Nachbar herstellen: Stück i umlegen
            wert = unterschied(gewichte, gruppe)
            gruppe[i] = 1 - gruppe[i]                   # und gleich wieder zurücklegen
            if wert < bester:
                bester = wert
                bester_teil = i
        if bester_teil == -1:                           # kein Nachbar ist besser: fertig
            break
        gruppe[bester_teil] = 1 - gruppe[bester_teil]   # zum besten Nachbarn gehen
        schritte = schritte + 1
    return gruppe, schritte

import random

def neustarts(gewichte, versuche, startwert):
    zufall = random.Random(startwert)                   # EINMAL, vor allen Versuchen
    bester = sum(gewichte)
    for versuch in range(versuche):
        start = [zufall.randint(0, 1) for teil in gewichte]
        ende, schritte = bergauf(gewichte, start)
        wert = unterschied(gewichte, ende)
        if wert < bester:
            bester = wert
    return bester

for versuche in [1, 3, 10, 30]:
    werte = [neustarts(gewichte, versuche, startwert) for startwert in range(8)]
    print(versuche, 'Neustarts:', werte)
```

Mit **einem** Versuch schwankt das Ergebnis stark (von 2 bis 14), mit **3** liegt es nur noch bei 0 bis 4, mit **10** bei 0 bis 2. Mit **30** Neustarts wird bei allen acht Startwerten der Unterschied **0**
gefunden. Das ist das Optimum, das wir vorhin durch Durchprobieren kennengelernt haben. Die Zahlen hängen vom Startwert ab: Mit anderen Startwerten kämen andere heraus, aber
das Bild bliebe gleich.

</details>

#### Frage 2.3

1. Warum ist das Ergebnis mit 10 Neustarts bei **jedem** Startwert mindestens so gut wie mit 3 Neustarts? (Hinweis: Beide fangen mit denselben Zufallszahlen an.)
2. Wie viele Bewertungen (Aufrufe von `unterschied`) braucht ein Versuch ungefähr, wenn das Hill-Climbing **s** Schritte macht? Wie viele sind es bei 30 Neustarts mit je etwa 5 Schritten?
3. Ist ein Zufallsneustart eine **Garantie**, das Optimum zu finden? Wie würdest du das in der Dokumentation ausdrücken?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Mit demselben Startwert liefert `random.Random` dieselbe Zahlenfolge. Die ersten 3 Versuche der 10 sind genau die 3 Versuche des kürzeren Laufs. Der kleinste Wert von mehr Versuchen kann daher nur **gleich oder kleiner** sein.
2. Jeder der `s` Schritte bewertet 14 Nachbarn, am Ende kommt noch ein Durchgang ohne Verbesserung dazu: etwa `(s + 1) · 14` Bewertungen. Bei 5 Schritten sind das rund 85, bei 30 Neustarts etwa **2 600**. Das ist viel weniger als 16 384
   und bei 40 Stücken ein winziger Bruchteil von 2^40.
3. Nein. Bei 30 Neustarts war es bei diesen acht Startwerten so, aber ein Startwert könnte Pech haben. Schreibe zum Beispiel: „Bestes gefundenes Ergebnis: Unterschied 0 bei 30 Neustarts, Startwert 3.
   Dass das Optimum erreicht ist, wurde hier durch Durchprobieren bestätigt. Bei größeren Eingaben gibt es dafür keine Garantie.“

</details>

#### Aufgabe 2.4

**Finde den Fehler.** Jemand hat `neustarts` so geschrieben (`bergauf` und die anderen Funktionen sind fertig). Die Ausgabe sieht seltsam aus: Egal, wie viele Neustarts, die Zahlen werden **nicht besser**.
Warum? Repariere die Funktion, die Ausgabe ist die der reparierten.

**Erwartete Ausgabe:**
```
1 Neustarts: [4, 2, 2, 10]
3 Neustarts: [2, 0, 2, 2]
10 Neustarts: [0, 0, 2, 2]
30 Neustarts: [0, 0, 0, 0]
```

In [ ]:
gewichte = [25, 47, 44, 18, 33, 48, 40, 50, 47, 14, 48, 10, 40, 26]

def unterschied(gewichte, gruppe):
    wagen0 = 0
    wagen1 = 0
    for i in range(len(gewichte)):
        if gruppe[i] == 0:
            wagen0 = wagen0 + gewichte[i]
        else:
            wagen1 = wagen1 + gewichte[i]
    return abs(wagen0 - wagen1)

def bergauf(gewichte, gruppe):
    gruppe = gruppe[:]                                  # Kopie, die Liste des Aufrufers bleibt unverändert
    schritte = 0
    while True:
        bester = unterschied(gewichte, gruppe)
        bester_teil = -1                                # -1: noch kein besserer Nachbar gefunden
        for i in range(len(gewichte)):
            gruppe[i] = 1 - gruppe[i]                   # Nachbar herstellen: Stück i umlegen
            wert = unterschied(gewichte, gruppe)
            gruppe[i] = 1 - gruppe[i]                   # und gleich wieder zurücklegen
            if wert < bester:
                bester = wert
                bester_teil = i
        if bester_teil == -1:                           # kein Nachbar ist besser: fertig
            break
        gruppe[bester_teil] = 1 - gruppe[bester_teil]   # zum besten Nachbarn gehen
        schritte = schritte + 1
    return gruppe, schritte

import random

def neustarts_falsch(gewichte, versuche, startwert):
    bester = sum(gewichte)
    for versuch in range(versuche):
        zufall = random.Random(startwert)               # neuer Zufallsgenerator in jedem Versuch
        start = [zufall.randint(0, 1) for teil in gewichte]
        ende, schritte = bergauf(gewichte, start)
        wert = unterschied(gewichte, ende)
        if wert < bester:
            bester = wert
    return bester

for versuche in [1, 3, 10, 30]:
    print(versuche, 'Neustarts:', [neustarts_falsch(gewichte, versuche, startwert) for startwert in range(4)])

<details>
<summary>Tipp anzeigen</summary>

Was liefert `random.Random(startwert)` bei jedem Aufruf? Und was liefert dann `randint` im ersten Aufruf?

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
gewichte = [25, 47, 44, 18, 33, 48, 40, 50, 47, 14, 48, 10, 40, 26]

def unterschied(gewichte, gruppe):
    wagen0 = 0
    wagen1 = 0
    for i in range(len(gewichte)):
        if gruppe[i] == 0:
            wagen0 = wagen0 + gewichte[i]
        else:
            wagen1 = wagen1 + gewichte[i]
    return abs(wagen0 - wagen1)

def bergauf(gewichte, gruppe):
    gruppe = gruppe[:]                                  # Kopie, die Liste des Aufrufers bleibt unverändert
    schritte = 0
    while True:
        bester = unterschied(gewichte, gruppe)
        bester_teil = -1                                # -1: noch kein besserer Nachbar gefunden
        for i in range(len(gewichte)):
            gruppe[i] = 1 - gruppe[i]                   # Nachbar herstellen: Stück i umlegen
            wert = unterschied(gewichte, gruppe)
            gruppe[i] = 1 - gruppe[i]                   # und gleich wieder zurücklegen
            if wert < bester:
                bester = wert
                bester_teil = i
        if bester_teil == -1:                           # kein Nachbar ist besser: fertig
            break
        gruppe[bester_teil] = 1 - gruppe[bester_teil]   # zum besten Nachbarn gehen
        schritte = schritte + 1
    return gruppe, schritte

import random

def neustarts_falsch(gewichte, versuche, startwert):
    zufall = random.Random(startwert)                   # EINMAL, vor allen Versuchen
    bester = sum(gewichte)
    for versuch in range(versuche):
        start = [zufall.randint(0, 1) for teil in gewichte]
        ende, schritte = bergauf(gewichte, start)
        wert = unterschied(gewichte, ende)
        if wert < bester:
            bester = wert
    return bester

for versuche in [1, 3, 10, 30]:
    print(versuche, 'Neustarts:', [neustarts_falsch(gewichte, versuche, startwert) for startwert in range(4)])
```

Im fehlerhaften Programm entsteht in **jedem** Versuch ein **neuer** Zufallsgenerator mit **demselben** Startwert. Jeder Versuch beginnt dadurch mit denselben Zufallszahlen, also mit
**derselben** Startverteilung und liefert **dasselbe** Ergebnis. 30 Versuche sind dann nicht besser als einer. Richtig ist: `random.Random(startwert)` **einmal** vor der Schleife, dann liefert jeder
Aufruf von `randint` die **nächste** Zahl der Folge, und die Versuche unterscheiden sich.

</details>

## 3. Simulated Annealing: auch Verschlechterungen erlauben

Beim Hill-Climbing sitzt man im Tal fest, sobald alle Nachbarn höher liegen. Das **Simulated Annealing** (simuliertes Abkühlen, wie beim langsamen Abkühlen von Metall) erlaubt manchmal **Schritte nach oben**:

- In jedem Schritt wird **ein zufälliger Nachbar** gewählt (kein Durchsuchen aller Nachbarn).
- Ist er **nicht schlechter**, wird er angenommen.
- Ist er **schlechter** (um `delta = neu - aktuell > 0`), wird er trotzdem mit der Wahrscheinlichkeit `exp(-delta / T)` angenommen, sonst abgelehnt. Dabei ist `T` die **Temperatur** und `exp` die
  Exponentialfunktion aus `math` (`math.exp`).
- Nach jedem Schritt wird die Temperatur **kleiner**: `T = T * faktor` mit `faktor` knapp unter 1 (der **Abkühlplan**).

Warum `exp(-delta / T)`? Der Wert liegt zwischen 0 und 1 (er kann als Wahrscheinlichkeit dienen), und er ist **groß bei kleinem `delta` und hohem `T`** und **klein bei großem `delta` und niedrigem `T`**. Am Anfang (heiß) läuft man
fast frei herum, später (kalt) nimmt man fast nur noch Verbesserungen an. Das sieht man an einer Verschlechterung um 10 bei verschiedenen Temperaturen:

In [ ]:
import math

for T in [100, 10, 1]:
    print('T =', T, ': Wahrscheinlichkeit', '%.5f' % math.exp(-10 / T))

Bei `T = 100` wird eine Verschlechterung um 10 mit rund 90 % angenommen, bei `T = 10` mit rund 37 %, bei `T = 1` fast nie (0,005 %).

#### Aufgabe 3.1

Schreibe `abkuehlen(gewichte, schritte, temperatur, faktor, startwert)`. Sie gibt `(bester, aktuell)` zurück: den **besten** Unterschied, der unterwegs vorkam, und den Unterschied am **Ende**. Das Gerüst enthält
die zufällige Startverteilung und den zufälligen Nachbarn (Stück `i` wird umgelegt). Du schreibst, was danach kommt:
**annehmen** (`aktuell = neu`), wenn `neu <= aktuell` ist **oder** `zufall.random() < math.exp((aktuell - neu) / temperatur)`; sonst die Änderung **zurücknehmen**.
Merke außerdem den besten Wert und kühle am Ende jedes Schritts ab. Gestartet wird mit `T = 30`, `faktor = 0,995`, 2000 Schritten.

**Erwartete Ausgabe:**
```
Startwert 0 : bester Wert 0 , am Ende 2
Startwert 1 : bester Wert 2 , am Ende 10
Startwert 2 : bester Wert 2 , am Ende 4
Startwert 3 : bester Wert 2 , am Ende 4
Startwert 4 : bester Wert 0 , am Ende 0
Startwert 5 : bester Wert 2 , am Ende 4
Startwert 6 : bester Wert 2 , am Ende 4
Startwert 7 : bester Wert 0 , am Ende 4
```

In [ ]:
gewichte = [25, 47, 44, 18, 33, 48, 40, 50, 47, 14, 48, 10, 40, 26]

def unterschied(gewichte, gruppe):
    wagen0 = 0
    wagen1 = 0
    for i in range(len(gewichte)):
        if gruppe[i] == 0:
            wagen0 = wagen0 + gewichte[i]
        else:
            wagen1 = wagen1 + gewichte[i]
    return abs(wagen0 - wagen1)

import math
import random

def abkuehlen(gewichte, schritte, temperatur, faktor, startwert):
    zufall = random.Random(startwert)
    gruppe = [zufall.randint(0, 1) for teil in gewichte]       # zufällige Startverteilung
    aktuell = unterschied(gewichte, gruppe)
    bester = aktuell
    for schritt in range(schritte):
        i = zufall.randint(0, len(gewichte) - 1)               # zufälliger Nachbar
        gruppe[i] = 1 - gruppe[i]
        neu = unterschied(gewichte, gruppe)
        # dein Code: annehmen oder zurücknehmen, bester merken, abkühlen
    return bester, aktuell

for startwert in range(8):
    bester, ende = abkuehlen(gewichte, 2000, 30, 0.995, startwert)
    print('Startwert', startwert, ': bester Wert', bester, ', am Ende', ende)

<details>
<summary>Tipp anzeigen</summary>

`if neu <= aktuell or zufall.random() < math.exp((aktuell - neu) / temperatur):` dann `aktuell = neu`, sonst `gruppe[i] = 1 - gruppe[i]`. Danach `if aktuell < bester: bester = aktuell` und `temperatur = temperatur * faktor`.
Die Reihenfolge der Bedingung ist wichtig: Bei `neu <= aktuell` wird der rechte Teil von `or` nicht mehr ausgewertet.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
gewichte = [25, 47, 44, 18, 33, 48, 40, 50, 47, 14, 48, 10, 40, 26]

def unterschied(gewichte, gruppe):
    wagen0 = 0
    wagen1 = 0
    for i in range(len(gewichte)):
        if gruppe[i] == 0:
            wagen0 = wagen0 + gewichte[i]
        else:
            wagen1 = wagen1 + gewichte[i]
    return abs(wagen0 - wagen1)

import math
import random

def abkuehlen(gewichte, schritte, temperatur, faktor, startwert):
    zufall = random.Random(startwert)
    gruppe = [zufall.randint(0, 1) for teil in gewichte]       # zufällige Startverteilung
    aktuell = unterschied(gewichte, gruppe)
    bester = aktuell
    for schritt in range(schritte):
        i = zufall.randint(0, len(gewichte) - 1)               # zufälliger Nachbar
        gruppe[i] = 1 - gruppe[i]
        neu = unterschied(gewichte, gruppe)
        if neu <= aktuell or zufall.random() < math.exp((aktuell - neu) / temperatur):
            aktuell = neu                                      # annehmen
        else:
            gruppe[i] = 1 - gruppe[i]                          # ablehnen: die Änderung zurücknehmen
        if aktuell < bester:
            bester = aktuell
        temperatur = temperatur * faktor                       # abkühlen
    return bester, aktuell

for startwert in range(8):
    bester, ende = abkuehlen(gewichte, 2000, 30, 0.995, startwert)
    print('Startwert', startwert, ': bester Wert', bester, ', am Ende', ende)
```

Alle acht Läufe finden unterwegs einen Unterschied von höchstens 2, drei davon sogar das Optimum 0 (Startwerte 0, 4 und 7). Am **Ende** der Läufe liegt `aktuell` oft darüber (meist bei 4, bei Startwert 1 sogar bei 10):
Gegen Ende ist es kalt, und der Lauf kann sich nicht mehr aus seinem Tal befreien. Deshalb merkt man sich den **besten Wert unterwegs** (und bei einer richtigen Aufgabe auch die **dazugehörige Lösung**,
nicht nur den Wert).

</details>

#### Aufgabe 3.2

Wie wichtig ist der **Abkühlplan**? `abkuehlen` ist fertig. Vergleiche drei Pläne: **zu kalt** (`T = 0,001`), **passend** (`T = 30`, Faktor 0,995) und **immer heiß** (`T = 1000`, Faktor 1, es wird gar nicht abgekühlt).
Rufe `abkuehlen` jeweils für die Startwerte 0 bis 9 auf und addiere die besten Werte (`summe_beste`) und die Werte am Ende (`summe_ende`).

**Erwartete Ausgabe:**
```
T = 0.001 , Faktor 0.995 : Summe der besten Werte 100 , der Endwerte 100
T = 30 , Faktor 0.995 : Summe der besten Werte 12 , der Endwerte 36
T = 1000 , Faktor 1.0 : Summe der besten Werte 0 , der Endwerte 886
```

In [ ]:
gewichte = [25, 47, 44, 18, 33, 48, 40, 50, 47, 14, 48, 10, 40, 26]

def unterschied(gewichte, gruppe):
    wagen0 = 0
    wagen1 = 0
    for i in range(len(gewichte)):
        if gruppe[i] == 0:
            wagen0 = wagen0 + gewichte[i]
        else:
            wagen1 = wagen1 + gewichte[i]
    return abs(wagen0 - wagen1)

import math
import random

def abkuehlen(gewichte, schritte, temperatur, faktor, startwert):
    zufall = random.Random(startwert)
    gruppe = [zufall.randint(0, 1) for teil in gewichte]       # zufällige Startverteilung
    aktuell = unterschied(gewichte, gruppe)
    bester = aktuell
    for schritt in range(schritte):
        i = zufall.randint(0, len(gewichte) - 1)               # zufälliger Nachbar
        gruppe[i] = 1 - gruppe[i]
        neu = unterschied(gewichte, gruppe)
        if neu <= aktuell or zufall.random() < math.exp((aktuell - neu) / temperatur):
            aktuell = neu                                      # annehmen
        else:
            gruppe[i] = 1 - gruppe[i]                          # ablehnen: die Änderung zurücknehmen
        if aktuell < bester:
            bester = aktuell
        temperatur = temperatur * faktor                       # abkühlen
    return bester, aktuell

plaene = [(0.001, 0.995), (30, 0.995), (1000, 1.0)]       # (Starttemperatur, Faktor)
for temperatur, faktor in plaene:
    summe_beste = 0
    summe_ende = 0
    # dein Code: für die Startwerte 0 bis 9 abkuehlen(gewichte, 2000, temperatur, faktor, startwert) aufrufen
    print('T =', temperatur, ', Faktor', faktor, ': Summe der besten Werte', summe_beste, ', der Endwerte', summe_ende)

<details>
<summary>Tipp anzeigen</summary>

Innere Schleife `for startwert in range(10):` mit `bester, ende = abkuehlen(gewichte, 2000, temperatur, faktor, startwert)`, dann beide Werte aufsummieren.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
gewichte = [25, 47, 44, 18, 33, 48, 40, 50, 47, 14, 48, 10, 40, 26]

def unterschied(gewichte, gruppe):
    wagen0 = 0
    wagen1 = 0
    for i in range(len(gewichte)):
        if gruppe[i] == 0:
            wagen0 = wagen0 + gewichte[i]
        else:
            wagen1 = wagen1 + gewichte[i]
    return abs(wagen0 - wagen1)

import math
import random

def abkuehlen(gewichte, schritte, temperatur, faktor, startwert):
    zufall = random.Random(startwert)
    gruppe = [zufall.randint(0, 1) for teil in gewichte]       # zufällige Startverteilung
    aktuell = unterschied(gewichte, gruppe)
    bester = aktuell
    for schritt in range(schritte):
        i = zufall.randint(0, len(gewichte) - 1)               # zufälliger Nachbar
        gruppe[i] = 1 - gruppe[i]
        neu = unterschied(gewichte, gruppe)
        if neu <= aktuell or zufall.random() < math.exp((aktuell - neu) / temperatur):
            aktuell = neu                                      # annehmen
        else:
            gruppe[i] = 1 - gruppe[i]                          # ablehnen: die Änderung zurücknehmen
        if aktuell < bester:
            bester = aktuell
        temperatur = temperatur * faktor                       # abkühlen
    return bester, aktuell

plaene = [(0.001, 0.995), (30, 0.995), (1000, 1.0)]       # (Starttemperatur, Faktor)
for temperatur, faktor in plaene:
    summe_beste = 0
    summe_ende = 0
    for startwert in range(10):
        bester, ende = abkuehlen(gewichte, 2000, temperatur, faktor, startwert)
        summe_beste = summe_beste + bester
        summe_ende = summe_ende + ende
    print('T =', temperatur, ', Faktor', faktor, ': Summe der besten Werte', summe_beste, ', der Endwerte', summe_ende)
```

**Zu kalt:** Mit `T = 0,001` wird (fast) nie eine Verschlechterung angenommen, die Suche verhält sich wie Hill-Climbing mit zufälligen Nachbarn und bleibt im ersten Tal hängen (beide Summen sind 100, also im Mittel 10 je Lauf).
**Passend:** Anfangs wird viel erlaubt, später fast nichts mehr. Die Summe der besten Werte ist klein (12, also im Mittel etwa 1), die Endwerte liegen mit insgesamt 36 nicht weit davon entfernt.
**Immer heiß:** Die Suche läuft **wie bei einem Zufallsspaziergang** durch alle Verteilungen. Der Wert am **Ende** ist so schlecht wie ein beliebiger zufälliger Wert (Endsumme 886, im Mittel fast 89 pro Lauf).
Dass die **besten** Werte unterwegs trotzdem alle 0 sind, liegt nur daran, dass es bei 14 Stücken insgesamt 16 384 Verteilungen gibt und 2000 zufällige Schritte durch Zufall eine perfekte treffen. Bei 40 Stücken
(2^40 Verteilungen) wäre das **hoffnungslos**. Ein Abkühlplan sorgt dafür, dass die Suche am Ende **im guten Tal bleibt**.

</details>

#### Frage 3.3

1. Was passiert, wenn die Temperatur **steigt** statt zu sinken (zum Beispiel `temperatur = temperatur / 0,995`)?
2. Wie kann man die Anzahl der Schritte, die Starttemperatur und den Faktor **sinnvoll wählen**? Was ist ein vernünftiger Ansatz für die Starttemperatur?
3. Vergleiche Simulated Annealing mit Zufallsneustarts: Wo ist der Unterschied in der Arbeitsweise?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Die Verschlechterungen werden im Lauf der Zeit **immer häufiger** angenommen. Am Ende läuft die Suche ganz **frei umher** und liefert schlechte Werte. Das Verfahren ist dann schlechter als Hill-Climbing.
2. Die Starttemperatur sollte in der Größenordnung **typischer Verschlechterungen** liegen (hier: einzelne Stücke wiegen 10 bis 50 kg, also eine Starttemperatur von etwa 30), damit am Anfang vieles erlaubt ist. Der Faktor und die Schritte
   bestimmen, **wie lange** man abkühlt: Mit `0,995` und 2000 Schritten sinkt `T` auf `30 · 0,995^2000`, also fast auf 0. Das Probieren mehrerer Pläne (wie in 3.2) gehört zum Verfahren.
3. Bei den **Neustarts** laufen die Versuche unabhängig voneinander jeweils **bergab**. Beim **Annealing** gibt es **einen langen Lauf**, der sich anfangs frei bewegt und dann immer genauer sucht. Neustarts
   sind einfacher zu programmieren, Annealing kommt oft aus Tälern heraus, in denen Hill-Climbing stecken bleibt.

</details>

### Finde den Fehler: Änderung nicht zurückgenommen

#### Aufgabe 3.4

**Finde den Fehler.** Dieses Annealing meldet als Ergebnis `aktuell`. Zur Kontrolle wird am Ende der Unterschied der gespeicherten Verteilung `gruppe` **neu berechnet**. Was fällt dir auf? Repariere die Funktion,
die Ausgabe ist die der reparierten Funktion.

**Erwartete Ausgabe:**
```
gemeldet 2 , tatsächlich 2
gemeldet 10 , tatsächlich 10
gemeldet 4 , tatsächlich 4
gemeldet 4 , tatsächlich 4
```

In [ ]:
gewichte = [25, 47, 44, 18, 33, 48, 40, 50, 47, 14, 48, 10, 40, 26]

def unterschied(gewichte, gruppe):
    wagen0 = 0
    wagen1 = 0
    for i in range(len(gewichte)):
        if gruppe[i] == 0:
            wagen0 = wagen0 + gewichte[i]
        else:
            wagen1 = wagen1 + gewichte[i]
    return abs(wagen0 - wagen1)

import math
import random

def abkuehlen_falsch(gewichte, schritte, temperatur, faktor, startwert):
    zufall = random.Random(startwert)
    gruppe = [zufall.randint(0, 1) for teil in gewichte]
    aktuell = unterschied(gewichte, gruppe)
    for schritt in range(schritte):
        i = zufall.randint(0, len(gewichte) - 1)
        gruppe[i] = 1 - gruppe[i]                              # Stück umlegen
        neu = unterschied(gewichte, gruppe)
        if neu <= aktuell or zufall.random() < math.exp((aktuell - neu) / temperatur):
            aktuell = neu
        temperatur = temperatur * faktor
    return aktuell, unterschied(gewichte, gruppe)

for startwert in range(4):
    gemeldet, echt = abkuehlen_falsch(gewichte, 2000, 30, 0.995, startwert)
    print('gemeldet', gemeldet, ', tatsächlich', echt)

<details>
<summary>Tipp anzeigen</summary>

Was passiert mit `gruppe`, wenn der Nachbar **abgelehnt** wird? Vergleiche mit der Funktion `abkuehlen` aus 3.1.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
gewichte = [25, 47, 44, 18, 33, 48, 40, 50, 47, 14, 48, 10, 40, 26]

def unterschied(gewichte, gruppe):
    wagen0 = 0
    wagen1 = 0
    for i in range(len(gewichte)):
        if gruppe[i] == 0:
            wagen0 = wagen0 + gewichte[i]
        else:
            wagen1 = wagen1 + gewichte[i]
    return abs(wagen0 - wagen1)

import math
import random

def abkuehlen_falsch(gewichte, schritte, temperatur, faktor, startwert):
    zufall = random.Random(startwert)
    gruppe = [zufall.randint(0, 1) for teil in gewichte]
    aktuell = unterschied(gewichte, gruppe)
    for schritt in range(schritte):
        i = zufall.randint(0, len(gewichte) - 1)
        gruppe[i] = 1 - gruppe[i]                              # Stück umlegen
        neu = unterschied(gewichte, gruppe)
        if neu <= aktuell or zufall.random() < math.exp((aktuell - neu) / temperatur):
            aktuell = neu
        else:
            gruppe[i] = 1 - gruppe[i]                          # abgelehnt: zurücknehmen
        temperatur = temperatur * faktor
    return aktuell, unterschied(gewichte, gruppe)

for startwert in range(4):
    gemeldet, echt = abkuehlen_falsch(gewichte, 2000, 30, 0.995, startwert)
    print('gemeldet', gemeldet, ', tatsächlich', echt)
```

Ein abgelehnter Nachbar wird **nicht zurückgenommen**: Das Stück bleibt umgelegt, `gruppe` ändert sich also bei **jedem** Schritt, aber `aktuell` wird nur bei angenommenen Schritten neu gesetzt. Die Werte `aktuell` und der echte
Unterschied laufen auseinander, und das Programm meldet einen Wert, den **die gespeicherte Lösung gar nicht hat**. Solche Fehler fallen auf, wenn man das Ergebnis am Ende **unabhängig neu berechnet**
(das gehört in jede Heuristik: gemeldeter Wert und nachgerechneter Wert müssen übereinstimmen).

</details>

## 4. Ehrlich bleiben: wie gut ist das Ergebnis?

Eine Heuristik liefert **eine** Lösung, ohne zu sagen, wie nahe sie am Optimum ist. Für die **Dokumentation** in der 1. Runde heißt das:

#### Frage 4.1

Du hast für eine Wettbewerbsaufgabe ein Annealing programmiert. Welche **vier Dinge** schreibst du in die Dokumentation, damit ein Leser dein Ergebnis einordnen und nachprüfen kann?
Nenne außerdem **zwei Wege**, die **Qualität** des Ergebnisses einzuschätzen.

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

**Das gehört hinein:**

1. das **Ergebnis** selbst (die Lösung und ihr Wert), zum Beispiel „Unterschied 0“ oder „kleinster Abstand 14“,
2. das **Verfahren** mit Bewertung, Nachbarn und Abkühlplan (Starttemperatur, Faktor, Zahl der Schritte, Zahl der Neustarts),
3. der **Startwert** des Zufalls, damit jeder dieselben Zahlen bekommt,
4. eine **ehrliche Aussage über die Qualität**: Es ist eine Heuristik, **ohne Garantie**, dass es das Optimum ist.

**Qualität einschätzen:**

- Bei **kleinen** Eingaben mit **Durchprobieren** vergleichen (wie bei uns: Optimum 0, gefundener Wert 0 bis 2) und das Ergebnis dieses Vergleichs angeben.
- Eine **Schranke** berechnen, die man nicht unterschreiten kann: Ist die Gesamtmasse ungerade, ist der Unterschied mindestens 1. Erreicht das Ergebnis die Schranke, ist es **bewiesen optimal**.
- **Mehrere Startwerte** laufen lassen und die **Streuung** angeben: Wenn alle ungefähr gleich gut sind, spricht das für ein gutes Ergebnis.

</details>

## Wettbewerbs-Ecke 1: Schlüssellöcher

*Angelehnt an Aufgabe 4 „Schlüssellöcher“ der 1. Runde des 34. Bundeswettbewerbs Informatik (2015/16).*

Schlüsselkarten haben 5 × 5 Punkte, an jedem Punkt ist ein Loch oder kein Loch. Für ein Hotel sollen `N` **verschiedene** Schlüssel gemacht werden, die sich **möglichst stark unterscheiden**: Wenn ein
Loch verschmutzt, soll man den Schlüssel nicht mit einem anderen verwechseln. Das Maß für den Unterschied zweier Schlüssel ist die **Zahl der Punkte, an denen sie sich unterscheiden**
(**Hamming-Abstand**). Gut ist eine Schlüsselmenge, wenn der **kleinste** Abstand zwischen zwei Schlüsseln möglichst **groß** ist.

Die Aufgabe wird hier **vereinfacht**: Wir lassen das **Wenden** der Karten weg (in der echten Aufgabe darf kein Schlüssel das Spiegelbild eines anderen sein). Es gibt zu viele Mengen zum Durchprobieren
(`2^25` Muster, das sind 33 554 432, und `N` davon auswählen), also eine **lokale Suche**: Man startet mit `N` zufälligen Mustern und **dreht in jedem Schritt ein oder zwei zufällige Punkte** eines zufälligen Schlüssels um.
Ist die Menge danach **nicht schlechter**, behält man die Änderung.

Ein Muster ist eine Zahl mit 25 Bits (Punkt `k` = Bit `k`, 1 = Loch). Zwei Bits werden durch `a ^ b` verglichen und mit `bin(...).count('1')` gezählt. `x ^ (1 << k)` **dreht Bit `k` um**.

#### Aufgabe E.1: Was wird ausgegeben?

Überlege zuerst, was der Code ausgibt, und schreibe es auf. Führe die Zelle erst danach aus.

In [ ]:
a = 0b1010
b = 0b0110
print(bin(a ^ b))
print(bin(a ^ b).count('1'))
print(1 << 3)
print(bin(a ^ (1 << 3)))

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

**Ausgabe:**
```
0b1100
2
8
0b10
```

`a ^ b` hat an den Stellen eine 1, an denen sich `a` und `b` unterscheiden: `1010` und `0110` unterscheiden sich an den beiden **mittleren** Stellen (`0b1100`), und `.count('1')` zählt zwei Stellen.
`1 << 3` ist 8 (eine 1, um 3 Stellen nach links geschoben), und `a ^ (1 << 3)` dreht genau das Bit mit Wert 8 um: Aus `0b1010` wird `0b10` (das Bit 3 wird gelöscht).

</details>

#### Aufgabe E.2

`abstand` und `bewertung` sind fertig. Schreibe die Schleife in `suche(N, schritte, startwert)`: Wähle einen zufälligen Schlüssel `i`, drehe bei einer Kopie einen zufälligen Punkt um (`^ (1 << zufälliges Bit)`), mit
Wahrscheinlichkeit 0,3 noch einen zweiten. Ist das neue Muster schon in der Liste, überspringe den Schritt (`continue`). Sonst bewertest du die geänderte Menge und **behältst** sie, wenn sie **nicht schlechter** ist
(`wert >= beste`; Tupel werden von links nach rechts verglichen). Die Ausgabe zeigt die Ergebnisse für `N = 7` (5000 Schritte) und `N = 20` (10 000 Schritte).

**Erwartete Ausgabe:**
```
N = 7 , Startwert 0 : kleinster Abstand 14 , Paare mit diesem Abstand 15 von 21
N = 7 , Startwert 1 : kleinster Abstand 14 , Paare mit diesem Abstand 18 von 21
N = 7 , Startwert 2 : kleinster Abstand 14 , Paare mit diesem Abstand 15 von 21
N = 20 , Startwert 0 : kleinster Abstand 12 , Paare mit diesem Abstand 97 von 190
N = 20 , Startwert 1 : kleinster Abstand 12 , Paare mit diesem Abstand 106 von 190
```

In [ ]:
import random

def abstand(a, b):
    return bin(a ^ b).count('1')                     # Zahl der Punkte, an denen sich a und b unterscheiden

def bewertung(schluessel):
    """(kleinster Abstand, minus Zahl der Paare mit diesem Abstand). Größer ist besser."""
    kleinster = 99
    anzahl = 0
    for i in range(len(schluessel)):
        for j in range(i):
            d = abstand(schluessel[i], schluessel[j])
            if d < kleinster:
                kleinster = d
                anzahl = 1
            elif d == kleinster:
                anzahl = anzahl + 1
    return (kleinster, -anzahl)                      # Tupel werden der Reihe nach verglichen

def suche(N, schritte, startwert):
    zufall = random.Random(startwert)
    S = []
    while len(S) < N:                                # N verschiedene zufällige Muster
        x = zufall.randint(0, 2 ** 25 - 1)
        if x not in S:
            S.append(x)
    beste = bewertung(S)
    for schritt in range(schritte):
        pass                                         # dein Code
    return S, beste

for N, schritte, startwert in [(7, 5000, 0), (7, 5000, 1), (7, 5000, 2), (20, 10000, 0), (20, 10000, 1)]:
    S, beste = suche(N, schritte, startwert)
    print('N =', N, ', Startwert', startwert, ': kleinster Abstand', beste[0], ', Paare mit diesem Abstand', -beste[1], 'von', N * (N - 1) // 2)

<details>
<summary>Tipp anzeigen</summary>

Innerhalb der Schleife: `i = zufall.randint(0, N - 1)`, `x = S[i] ^ (1 << zufall.randint(0, 24))`, dann `if zufall.random() < 0.3:` den zweiten Punkt. Eine Kopie `neu = S[:]` bekommt `neu[i] = x`;
nur wenn `bewertung(neu) >= beste`, werden `S` und `beste` ersetzt.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
import random

def abstand(a, b):
    return bin(a ^ b).count('1')                     # Zahl der Punkte, an denen sich a und b unterscheiden

def bewertung(schluessel):
    """(kleinster Abstand, minus Zahl der Paare mit diesem Abstand). Größer ist besser."""
    kleinster = 99
    anzahl = 0
    for i in range(len(schluessel)):
        for j in range(i):
            d = abstand(schluessel[i], schluessel[j])
            if d < kleinster:
                kleinster = d
                anzahl = 1
            elif d == kleinster:
                anzahl = anzahl + 1
    return (kleinster, -anzahl)                      # Tupel werden der Reihe nach verglichen

def suche(N, schritte, startwert):
    zufall = random.Random(startwert)
    S = []
    while len(S) < N:                                # N verschiedene zufällige Muster
        x = zufall.randint(0, 2 ** 25 - 1)
        if x not in S:
            S.append(x)
    beste = bewertung(S)
    for schritt in range(schritte):
        i = zufall.randint(0, N - 1)                 # ein zufälliger Schlüssel
        x = S[i] ^ (1 << zufall.randint(0, 24))      # ein Punkt umgedreht
        if zufall.random() < 0.3:
            x = x ^ (1 << zufall.randint(0, 24))     # manchmal noch ein zweiter
        if x in S:
            continue                                 # doppelte Schlüssel sind verboten
        neu = S[:]
        neu[i] = x
        wert = bewertung(neu)
        if wert >= beste:                            # nicht schlechter: behalten
            S = neu
            beste = wert
    return S, beste

for N, schritte, startwert in [(7, 5000, 0), (7, 5000, 1), (7, 5000, 2), (20, 10000, 0), (20, 10000, 1)]:
    S, beste = suche(N, schritte, startwert)
    print('N =', N, ', Startwert', startwert, ': kleinster Abstand', beste[0], ', Paare mit diesem Abstand', -beste[1], 'von', N * (N - 1) // 2)
```

Bei **7** Schlüsseln wird bei allen drei Startwerten ein kleinster Abstand von **14** gefunden (nur die Zahl der Paare mit diesem Abstand schwankt). Bei **20** Schlüsseln ist der kleinste Abstand **12**.
Das stimmt mit dem [Lösungsnotebook zur echten Aufgabe](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf1/34_Schluesselloecher/schluesselloecher_loesung.ipynb) überein; dort wird mit Hilfe einer Tabelle aus den Lösungshinweisen gezeigt, dass 14 bzw. 12
**nicht mehr zu verbessern** sind. Wir haben also bei `N = 7` und `N = 20` ein **optimales** Ergebnis erreicht, aber **ohne Garantie**: Aus der Suche allein wüssten wir das nicht.

</details>

#### Frage E.3

1. Warum startet die Suche mit **zufälligen** Mustern? Was ist besser, wenn man nur wenig Zeit hat: **viele** Schritte von einem Start aus oder **wenige** Schritte mit vielen Neustarts?
2. In `suche` bleibt eine Änderung auch bei **gleicher** Bewertung (`>=` statt `>`) erhalten. Warum könnte das helfen?
3. Für `N = 1000` Schlüssel (wie in der echten Aufgabe) wäre ein Schritt viel teurer. Wie viele Paare muss `bewertung` dann bei jedem Schritt prüfen? Wie könnte man das beschleunigen?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Zufällige Muster unterscheiden sich im Mittel an etwa der Hälfte der Punkte (rund 12 von 25), sind also schon ein vernünftiger Start. Ob man **lange** sucht oder **oft** neu startet, lässt sich nicht allgemein sagen:
   Gibt es viele gleich gute Täler, helfen Neustarts, sonst eine lange Suche (wie unser Annealing in Abschnitt 3). Probieren mit mehreren Startwerten zeigt es.
2. Bei gleicher Bewertung ändert sich die Menge, ohne besser zu werden. So kann die Suche über **Plateaus** (Gebiete gleicher Bewertung) wandern und kommt vielleicht zu einer Stelle, an der eine Verbesserung möglich ist.
   Wer nur strikt Besseres annimmt, bleibt dort stehen, wo gerade nichts besser wird.
3. `N · (N - 1) / 2 = 499 500` Paare, bei jedem Schritt. Man kann **nur die Paare mit dem geänderten Schlüssel** neu berechnen (999 statt 499 500) und sich den **kleinsten Abstand** merken. Bei 1000 Schlüsseln
   hilft die lokale Suche trotzdem nicht mehr gut, dort nimmt man einen **bekannten Code** (siehe Lösungsnotebook).

</details>

## Wettbewerbs-Ecke 2: Wortsuche

*Angelehnt an Aufgabe 3 „Wortsuche“ der 1. Runde des 40. Bundeswettbewerbs Informatik (2021/22).*

In ein Rechteck aus Buchstaben sollen **Wörter versteckt** werden (wie im Rätselheft). Die Aufgabe wird **vereinfacht**: Die Wörter stehen nur nach **rechts**, nach **unten** oder **schräg nach rechts unten**,
nicht rückwärts, und das Auffüllen mit Buchstaben lassen wir weg (leere Felder bleiben `.`). Wörter dürfen sich **kreuzen**, wenn sie dort denselben Buchstaben haben.

Das Setzen geht oft **schief**: Ein Wort findet nach 20 zufälligen Versuchen keinen Platz mehr. Dann beginnt man das **ganze Rätsel von vorn**, mit anderen Zufallszahlen: ein **Zufallsneustart**.
Die Zahl der Versuche ist dabei eine **Messgröße** statt der Zeit.

Die Dateien `daten/wortsuche1.txt` (6 × 6) und `daten/wortsuche2.txt` (10 × 10) haben in der 1. Zeile Höhe und Breite, in der 2. die Zahl der Wörter, dann je Zeile ein Wort. Die Funktion `platziere` ist fertig:
Sie versucht, **alle** Wörter (die längsten zuerst) zu setzen, und gibt das Rechteck zurück oder **`None`**, wenn ein Wort nach 20 Versuchen keinen Platz fand.

#### Aufgabe E.4

Schreibe `raetsel(h, b, woerter, startwert, max_versuche)`: Sie erzeugt **einmal** `zufall = random.Random(startwert)` und ruft dann bis zu `max_versuche`-mal `platziere` auf. Beim ersten Erfolg gibt sie
`(gitter, nummer_des_versuchs)` zurück (Nummer ab 1), sonst `(None, max_versuche)`. Die Ausgabe zeigt ein 4 × 4-Rätsel mit den sechs Wörtern aus `wortsuche1.txt` (EIN, DA, ER, INFO, DU, UND),
die Zahl der Versuche bei acht Startwerten sowie bei 10 × 10 mit `wortsuche2.txt` und zum Schluss einen **Fall, der nie klappt**.

**Erwartete Ausgabe:**
```
.UND
INFO
EIND
RDAU
Startwert 2: 6 Versuche
4 x 4, Startwerte 0 bis 7: [1, 1, 6, 4, 1, 2, 1, 6]
10 x 10, Startwerte 0 bis 7: [5, 2, 2, 4, 6, 1, 5, 5]
9 x 9: None 500
```

In [ ]:
import random

def lies_woerter(dateiname):
    with open(dateiname, encoding='utf-8') as datei:
        zeilen = [zeile.strip() for zeile in datei if zeile.strip()]
    h, b = zeilen[0].split()
    n = int(zeilen[1])
    return int(h), int(b), [wort.upper() for wort in zeilen[2:2 + n]]

def platziere(h, b, woerter, zufall):
    """Ein Versuch: Gibt das Rechteck zurück (Liste von Zeilen, '.' = leer) oder None."""
    gitter = [['.'] * b for zeile in range(h)]
    for wort in sorted(woerter, key=lambda w: len(w), reverse=True):          # die längsten zuerst
        gesetzt = False
        for versuch in range(20):
            dx, dy = zufall.choice([(1, 0), (0, 1), (1, 1)])     # rechts, unten, schräg
            i = zufall.randint(0, h - 1)
            j = zufall.randint(0, b - 1)
            if i + dy * (len(wort) - 1) >= h or j + dx * (len(wort) - 1) >= b:
                continue                                         # ragt aus dem Rechteck
            passt = True
            for k in range(len(wort)):
                if gitter[i + dy * k][j + dx * k] not in ('.', wort[k]):
                    passt = False
            if passt:
                for k in range(len(wort)):
                    gitter[i + dy * k][j + dx * k] = wort[k]
                gesetzt = True
                break
        if not gesetzt:
            return None
    return gitter

def raetsel(h, b, woerter, startwert, max_versuche):
    zufall = random.Random(startwert)                 # EINMAL, vor allen Versuchen
    # dein Code
    return None, max_versuche

h, b, woerter = lies_woerter('daten/wortsuche1.txt')
gitter, versuche = raetsel(4, 4, woerter, 2, 500)
if gitter is not None:
    for zeile in gitter:
        print(''.join(zeile))
print('Startwert 2:', versuche, 'Versuche')

print('4 x 4, Startwerte 0 bis 7:', [raetsel(4, 4, woerter, startwert, 500)[1] for startwert in range(8)])
h, b, woerter = lies_woerter('daten/wortsuche2.txt')
print('10 x 10, Startwerte 0 bis 7:', [raetsel(h, b, woerter, startwert, 500)[1] for startwert in range(8)])
gitter, versuche = raetsel(9, 9, woerter, 0, 500)
print('9 x 9:', gitter, versuche)

<details>
<summary>Tipp anzeigen</summary>

`for versuch in range(1, max_versuche + 1):` Darin `gitter = platziere(h, b, woerter, zufall)`. Ist `gitter` nicht `None`, `return gitter, versuch`. Nach der Schleife `return None, max_versuche`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
import random

def lies_woerter(dateiname):
    with open(dateiname, encoding='utf-8') as datei:
        zeilen = [zeile.strip() for zeile in datei if zeile.strip()]
    h, b = zeilen[0].split()
    n = int(zeilen[1])
    return int(h), int(b), [wort.upper() for wort in zeilen[2:2 + n]]

def platziere(h, b, woerter, zufall):
    """Ein Versuch: Gibt das Rechteck zurück (Liste von Zeilen, '.' = leer) oder None."""
    gitter = [['.'] * b for zeile in range(h)]
    for wort in sorted(woerter, key=lambda w: len(w), reverse=True):          # die längsten zuerst
        gesetzt = False
        for versuch in range(20):
            dx, dy = zufall.choice([(1, 0), (0, 1), (1, 1)])     # rechts, unten, schräg
            i = zufall.randint(0, h - 1)
            j = zufall.randint(0, b - 1)
            if i + dy * (len(wort) - 1) >= h or j + dx * (len(wort) - 1) >= b:
                continue                                         # ragt aus dem Rechteck
            passt = True
            for k in range(len(wort)):
                if gitter[i + dy * k][j + dx * k] not in ('.', wort[k]):
                    passt = False
            if passt:
                for k in range(len(wort)):
                    gitter[i + dy * k][j + dx * k] = wort[k]
                gesetzt = True
                break
        if not gesetzt:
            return None
    return gitter

def raetsel(h, b, woerter, startwert, max_versuche):
    zufall = random.Random(startwert)                 # EINMAL, vor allen Versuchen
    for versuch in range(1, max_versuche + 1):
        gitter = platziere(h, b, woerter, zufall)
        if gitter is not None:
            return gitter, versuch
    return None, max_versuche

h, b, woerter = lies_woerter('daten/wortsuche1.txt')
gitter, versuche = raetsel(4, 4, woerter, 2, 500)
if gitter is not None:
    for zeile in gitter:
        print(''.join(zeile))
print('Startwert 2:', versuche, 'Versuche')

print('4 x 4, Startwerte 0 bis 7:', [raetsel(4, 4, woerter, startwert, 500)[1] for startwert in range(8)])
h, b, woerter = lies_woerter('daten/wortsuche2.txt')
print('10 x 10, Startwerte 0 bis 7:', [raetsel(h, b, woerter, startwert, 500)[1] for startwert in range(8)])
gitter, versuche = raetsel(9, 9, woerter, 0, 500)
print('9 x 9:', gitter, versuche)
```

Im 4 × 4-Rätsel klappt es bei Startwert 2 erst im **6. Versuch** (fünfmal neu begonnen), bei manchen Startwerten schon im ersten. Bei 10 × 10 sind es zwischen 1 und 6 Versuche.
Das Setzen scheitert also oft, aber **in wenigen Neustarts** klappt es. Bei 9 × 9 findet sich **nie** eine Lösung: `BILDSCHIRM` hat 10 Buchstaben, passt aber in kein 9 × 9-Rechteck. Die 500 Versuche waren umsonst.

</details>

#### Frage E.5

1. Bei 9 × 9 lief `raetsel` 500-mal vergeblich. Wie kann man **vor** den Neustarts erkennen, dass es nie klappt? Warum ist es wichtig, die Zahl der Neustarts zu **begrenzen**?
2. Für ein Rätsel mit **mittlerem** Schwierigkeitsgrad kommt es nicht nur darauf an, **dass** es klappt, sondern auch auf **Eigenschaften** des Ergebnisses (zum Beispiel wenige zufällige zusätzliche Fundstellen).
   Wie könnte man Neustarts dafür nutzen?
3. Wie kann man die Zahl der Neustarts **senken**, ohne den Zufall aufzugeben?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Man prüft vorher einfache **notwendige Bedingungen**: Ist ein Wort länger als `max(h, b)`, hat es nirgends Platz, und alle Neustarts sind verschenkt (hier `BILDSCHIRM` mit 10 Buchstaben, Rechteck 9 × 9). Auch die
   **Gesamtzahl** der Buchstaben darf die Zahl der Felder nicht übersteigen. Die **Begrenzung** der Versuche sorgt dafür, dass das Programm auch bei unlösbaren Eingaben **endet** und eine klare Meldung gibt.
2. Man erzeugt viele Rätsel, **bewertet** jedes (etwa: wie viele Fast-Treffer, wie viele Wörter laufen rückwärts) und behält das, das dem gewünschten Grad am nächsten kommt. Das ist ein Zufallsneustart **mit Bewertung**, wie in Abschnitt 2.
3. Man setzt die **längsten Wörter zuerst** (wie hier), und wenn das Zufallsraten scheitert, zählt man **alle** noch möglichen Plätze auf und wählt dort einen zufällig, statt abzubrechen. So
   scheitert ein Versuch nur, wenn es wirklich keinen Platz gibt (so macht es das [Lösungsnotebook zur echten Aufgabe](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf1/40_Wortsuche/wortsuche_loesung.ipynb)).

</details>

## Das hast du gelernt

```python
import math
import random

# Lokale Suche (Hill-Climbing): immer zum besten Nachbarn, solange er besser ist
while True:
    ...                                   # alle Nachbarn bewerten (ändern, bewerten, zurücknehmen)
    if kein_nachbar_besser:
        break                             # lokales Optimum

# Zufallsneustart: Zufallsgenerator EINMAL vor den Versuchen
zufall = random.Random(startwert)
for versuch in range(versuche):
    start = ...                           # zufällige Startlösung
    ...                                   # verbessern, bestes Ergebnis merken

# Simulated Annealing
neu = ...                                 # zufälliger Nachbar
if neu <= aktuell or zufall.random() < math.exp((aktuell - neu) / temperatur):
    aktuell = neu                         # annehmen
else:
    ...                                   # ablehnen: Änderung zurücknehmen
temperatur = temperatur * faktor          # abkühlen
```

- Eine **Heuristik** braucht eine **Bewertungsfunktion** (ein Zahlenwert) und **Nachbarn** (kleine Änderungen einer Lösung). Sie liefert **gute, aber keine garantiert besten** Lösungen.
- **Hill-Climbing** bleibt in einem **lokalen Optimum** stecken, auch wenn man mit einer **gierigen** Startlösung beginnt.
- **Zufallsneustarts** (mit einem **einmal** gesetzten Startwert!) und **Simulated Annealing** (Verschlechterungen mit sinkender Wahrscheinlichkeit `exp(-delta / T)`) helfen heraus. Die **Temperatur** muss **sinken**, sonst läuft die Suche frei umher.
- Rechne das Ergebnis am Ende **neu nach** (gemeldeter und echter Wert müssen übereinstimmen), und mache **Rücknahme** und **Neustart** unmissverständlich im Code.
- In die **Dokumentation**: Ergebnis, Verfahren und Parameter, **Startwert**, und **ehrlich**, wie gut das Ergebnis ist. Bei kleinen Eingaben mit dem **Optimum durch Durchprobieren** vergleichen, sonst mit einer **Schranke** oder mehreren Startwerten.
- Statt der Laufzeit zählt man **Schritte, Bewertungen und Versuche**.

### Aufgaben der 1. Runde dazu

Bearbeite eine Aufgabe zuerst selbst und sieh dir das Lösungsnotebook erst danach an.

- [Schlüssellöcher](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf1/34_Schluesselloecher/schluesselloecher_loesung.ipynb) (34. BwInf): Mindestabstand mit lokaler Suche, wendesichere Muster, für sehr viele Schlüssel ein bekannter Code
- [Wortsuche](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf1/40_Wortsuche/wortsuche_loesung.ipynb) (40.): Schwierigkeitsstufen, Zufallsneustarts beim Setzen der Wörter, Kontrolle durch Suchen

Im nächsten Notebook (17) geht es um **Geometrie**: Vektoren, Schnittpunkte von Strecken und Abstände, und wie man mit Kommazahlen sicher vergleicht.